# Test your IOL-AI 2026 script on Colab

A template to **test your competition script before you submit it**. It runs your model on real [Linguini](https://huggingface.co/datasets/facebook/linguini) problems — same format as the competition — so you can see what your script produces and check its answers.

The notebook has three parts:

- **Setup** and **Load the test problems** — Colab scaffolding, you don't touch it.
- **Your script** — your model + prompt + parsing. This is the part you experiment with and upload as `script.py`.
- **Check** — look at the output and score it (only possible here, where the answers are known).

**To test YOUR own script, change three things in the "Your script" section:**

1. `MODEL_ID` → your model repo (or any model that fits the T4's 16 GB).
2. `SYSTEM` → your prompt (the main lever).
3. `parse_answers` → how you read the answers back (must match the format your prompt asks for).

Then run, watch the output stream live, and check the score — iterate on the prompt until you're happy.

**To turn it into a real submission, only three more things change:**

1. `MODEL_ID` points at `.` (your weights ship inside the submission repo) instead of a Hub name.
2. You read the platform's hidden test set at `/tmp/data/test.csv` instead of Linguini.
3. You write `submission.csv` instead of scoring here.

Set **Runtime → Change runtime type → T4 GPU**, then run the cells in order.

## Setup

Installs the loader and pins `numpy`. Colab loads an older `numpy` at startup, so this cell **restarts the runtime once** after installing, so the new versions load cleanly. After it restarts, just **carry on with the cells below** (start again from "Load the test problems"). If it doesn't restart on its own: Runtime → Restart session, then continue.

In [ ]:
# Colab has internet, so we install the AWQ loader (gptqmodel) + deps. gptqmodel is only
# used if your model is AWQ/GPTQ — it's harmless to keep for any other model, so leave this
# line as is. numpy is pinned so the upgrade stays consistent.
!pip install -q -U transformers accelerate datasets sacrebleu gptqmodel "numpy==2.2.6"

# Colab loaded an older numpy at startup; the install just replaced it on disk, so we
# restart the runtime once to load the new one. After the restart, carry on with the
# cells below (re-running this one is fine too — it won't restart a second time).
import numpy, IPython
if numpy.__version__ != "2.2.6":
    print("Installed — restarting. When it's back, continue with the cells below.", flush=True)
    IPython.Application.instance().kernel.do_shutdown(True)

## Load the test problems

Colab scaffolding — you don't change this. We load a few Linguini problems, each with `context`, `query`, and the reference `answer`. In a real submission the platform hands you the same `context` / `query` columns at `/tmp/data/test.csv` (without the answers — which is exactly why you test here first).

In [ ]:
from datasets import load_dataset

N_PROBLEMS = 4   # how many problems to test on — keep it small so it's quick and easy to read

# Colab: Linguini (public, has reference answers).
# Submission: df = pd.read_csv("/tmp/data/test.csv", dtype=str).fillna("")   # same context/query columns
ds = load_dataset("facebook/linguini")
df = ds["test"].to_pandas().sample(N_PROBLEMS, random_state=0).reset_index(drop=True)
print(len(df), "problems |", list(df["task_type"].unique()))

## Your script

Everything from here to **Check** is your submission — the model, the prompt, and how you read the answers back. This is the part you experiment with and upload as `script.py`. It takes each problem's `context` + `query` from `df`, asks the model, and parses its answer.

In [ ]:
import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Free a model already on the GPU, so re-running this cell doesn't stack a second copy and run out of
# memory. (If you still hit "out of memory", do Runtime -> Restart session and run this cell just once.)
model = tok = None
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# ===== CHANGE HERE — your model (must fit the T4's ~15 GB: AWQ/GPTQ, or a smaller fp16 model) =====
MODEL_ID = "ritaberrada/iol-qwen25-14b-awq"   # your Hub repo (becomes "." inside the submission)
MAX_NEW_TOKENS = 1536                          # room to reason; lower = faster but answers may get cut off

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto",
).eval()
print("loaded", MODEL_ID, "| VRAM", round(torch.cuda.max_memory_allocated() / 1e9, 1), "GB")

In [ ]:
import re

# ===== CHANGE HERE — your prompt (the main lever: how you ask the model) =====
SYSTEM = (
    "You solve International Linguistics Olympiad problems by reasoning from the "
    "data you are given. You may meet a task type you have never seen: read the "
    "instruction and the examples, and answer in the same form they use. "
    "Common task types and what to give -- "
    "translation: the translated form only, in the language the task asks for; "
    "fill_blanks: only the missing form for each blank; "
    "match_letters: only the option letter (for example A, B, C); "
    "text_to_num: the number in digits; "
    "num_to_text: the number written out in words, in the language asked; "
    "any other type: give exactly what the instruction asks, nothing else. "
    "Reason step by step first. Then write a line that says exactly FINAL ANSWERS: "
    "and, below it, one answer per line in the order the items are asked -- the "
    "bare answer only, no numbering, no quotes, no extra text."
)

# ===== CHANGE HERE — how you read the answers back (must match the format your prompt asks for) =====
def parse_answers(text):
    """Keep only the lines after the last 'FINAL ANSWERS:' marker, one answer per line."""
    marker = list(re.finditer(r"(?im)^\s*final answers?\s*:?\s*$", text))
    if marker:
        text = text[marker[-1].end():]
    answers = []
    for line in text.splitlines():
        line = re.sub(r"^\s*\d+[.)]\s*", "", line).strip()   # drop "1. " / "2) " if the model adds it
        if line:
            answers.append(line)
    return answers

In [ ]:
# Answer every problem, in order. We stream each answer live (TextStreamer) so the workshop can
# watch the model reason -- it takes ~1 min per problem. Streaming is display only: the model
# produces exactly the same text as the submission, we just show it as it comes.
from transformers import TextStreamer

results = []
for i, r in df.iterrows():
    print(f"\n{'=' * 72}\nPROBLEM {i + 1}/{len(df)} -- {r['task_type']}\n{'=' * 72}", flush=True)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{r['context'].strip()}\n\n{r['query'].strip()}"},
    ]
    # return_dict=True: recent transformers (on Colab) returns a dict here, not a bare tensor,
    # so we unpack it into generate. The sandbox's older transformers returns the tensor directly.
    enc = tok.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)   # prints tokens as they come
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, streamer=streamer)
    text = tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    answers = parse_answers(text)
    # A submission would instead collect {"id": r["id"], "pred": json.dumps(answers)} and write submission.csv.
    results.append({"query": r["query"], "reference": r["answer"], "raw": text, "pred": answers})
    print(f"\n--> parsed {len(answers)} answers", flush=True)

## Check your script (Colab only)

You just watched each answer stream live above. Now — because Linguini ships the reference answers — you can line up what your script parsed against the truth and score it, which the sandbox can't do.

In [ ]:
# What your script parsed, next to the reference answers (the reasoning already streamed above).
for i, res in enumerate(results, 1):
    print(f"PROBLEM {i} -- {len(res['pred'])} answers")
    print("  parsed:", res["pred"])
    print("  gold  :", res["reference"])
    print()

In [ ]:
import ast
from sacrebleu.metrics import CHRF
chrf = CHRF()

def gold_alts(reference):
    """Reference: a list of items, each a list of accepted alternatives."""
    v = ast.literal_eval(str(reference))
    return [[str(a) for a in it] if isinstance(it, (list, tuple)) else [str(it)] for it in v]

# Score each item first, then the overall average.
em_total = cf_total = n = 0
for pi, res in enumerate(results, 1):
    gold = gold_alts(res["reference"])
    preds = (res["pred"] + [""] * len(gold))[:len(gold)]   # line up by position, as the scorer does
    print(f"PROBLEM {pi}")
    for k, (p, alts) in enumerate(zip(preds, gold), 1):
        hit = any(p.strip().lower() == a.strip().lower() for a in alts)
        cf = max(chrf.sentence_score(p, [a]).score for a in alts)
        em_total += hit
        cf_total += cf
        n += 1
        print(f"  item {k}: EM {'OK' if hit else '--'}  chrF {cf:5.1f}   pred={p!r}  gold={alts}")
    print()

print("=" * 60)
print(f"FINAL   exact match {em_total / n:.2f}   |   chrF {cf_total / n:.1f}   ({n} items)")

## Your turn — the human challenge ✍️

The competition has a **human track** too: the model gave its answers above — now *you* explain the reasoning. Pick one of the problems, work out how the language actually works, and write it up in the last cell: the pattern you found, the evidence for it, and how each answer follows.

Use the cell below to look at a problem in full, then **double-click the explanation cell** to write yours. (This part is yours — nothing here is graded by the notebook.)

In [ ]:
# (optional) Show one problem in full so you can study the data while you write your explanation.
p = 0   # pick a problem: 0, 1, 2, or 3
print("CONTEXT:\n", df.iloc[p]["context"])
print("\nQUERY:\n", df.iloc[p]["query"])

### My explanation — the human challenge

*Double-click here to edit. Replace the prompts below with your own reasoning.*

**Problem I chose:** _(task type / language)_

**The pattern I found**
- _…_

**The evidence in the data**
- _…_

**How I got each answer**
1. _…_
2. _…_